# 04b — Pydantic AI: typed compliance caseworker

## Scenario: payment reviews with a machine-consumable decision

Northstar Commerce reviews cross-border merchant payments. The caseworker may recommend **approve**, **escalate**, or **reject**, but a model response must never directly move money or override policy. It must carry evidence IDs, a reason, and a review requirement that downstream code can validate.

This notebook uses deterministic fixtures locally, then maps them to Pydantic AI’s typed-agent model: dependencies, tools, dynamic instructions, output schemas, validation retries, instrumentation, and evaluation.

**Outcome:** build a decision contract that is useful to a workflow without confusing JSON validity with correctness or authorization.


## 1. Architecture: type safety is a control surface

![Pydantic AI compliance caseworker architecture](../../../assets/pydanticai-compliance-caseworker.svg)

Pydantic AI is especially useful when the central risk is turning an LLM response into a reliable Python business object. Its agent loop, dependency injection, tools, structured outputs, validation/retry behavior, provider flexibility, and observability make it a strong fit for typed application domains.

| Layer | What it proves | What it cannot prove |
| --- | --- | --- |
| Input model | fields have the expected shape/range | that the request is authorized |
| Tool schema | arguments match a narrow interface | that returned data is current or trustworthy |
| Output model | result has allowed values and required fields | that diagnosis or evidence is factually correct |
| Evidence gate | source IDs exist and belong to case scope | that sources support the conclusion |
| Policy service | deterministic thresholds and permissions are applied | that the model was useful |

See the official [Pydantic AI overview](https://pydantic.dev/docs/ai/overview/) and [agents guide](https://pydantic.dev/docs/ai/core-concepts/agent/) for current API details.


## 2. Step 1 — Define data contracts before the agent

Use Pydantic models at the boundaries. Constraints should encode stable business invariants; policy rules that change frequently should live in a policy service or configuration—not be buried in a model prompt.

```python
from typing import Literal
from pydantic import BaseModel, Field, field_validator, model_validator

class PaymentCase(BaseModel):
    case_id: str = Field(pattern=r"^case-[0-9]+$")
    tenant_id: str = Field(min_length=3)
    amount_usd: int = Field(gt=0, le=1_000_000)
    country: str = Field(min_length=2, max_length=2)

    @field_validator("country")
    @classmethod
    def normalize_country(cls, value: str) -> str:
        return value.upper()

class ComplianceDecision(BaseModel):
    decision: Literal["approve", "escalate", "reject"]
    evidence_ids: list[str] = Field(min_length=1)
    rationale: str = Field(min_length=20)
    requires_human_review: bool

    @model_validator(mode="after")
    def enforce_escalation_review(self):
        if self.decision == "escalate" and not self.requires_human_review:
            raise ValueError("Escalation must require human review")
        return self
```

Validation errors are valuable feedback, not a signal to silently coerce data. Record them safely and decide whether the request should be corrected, retried, or escalated.


In [ ]:
from pathlib import Path
import sys
course_dir=Path.cwd().resolve()
for candidate in (course_dir,*course_dir.parents):
    if (candidate/'curriculum'/'beginner'/'04-agent-development-frameworks').exists():
        course_dir=candidate/'curriculum'/'beginner'/'04-agent-development-frameworks'; break
else: raise RuntimeError('Run from a repository checkout.')
sys.path.insert(0,str(course_dir)) if str(course_dir) not in sys.path else None
from lab import pydanticai_shaped_compliance_case
low=pydanticai_shaped_compliance_case('case-100',450,'CA')
high=pydanticai_shaped_compliance_case('case-101',12_500,'CA')
print(low); print(high)
assert low.decision=='approve' and high.decision=='escalate'


## 3. Step 2 — Inject scoped dependencies, not globals

Dependencies make the source of authority explicit and make unit tests easier. Pass a case-scoped repository, tenant identity, policy version, and clock through a typed dependency object. Do not let a model choose a database connection, tenant ID, or policy version.

```python
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext

@dataclass
class ComplianceDeps:
    tenant_id: str
    case_id: str
    policy_version: str
    repository: "ComplianceRepository"

agent = Agent(
    "provider:model",  # choose provider/model in deployment configuration
    deps_type=ComplianceDeps,
    output_type=ComplianceDecision,
    instructions="Return a cited compliance recommendation. Never approve without evidence.",
)

@agent.instructions
async def add_case_scope(ctx: RunContext[ComplianceDeps]) -> str:
    return f"Evaluate only case {ctx.deps.case_id} for tenant {ctx.deps.tenant_id}."
```

Pydantic AI’s [dependency documentation](https://pydantic.dev/docs/ai/core-concepts/dependencies/) explains how typed dependencies are available to dynamic instructions and tools.


## 4. Step 3 — Give the model narrow, typed, read-only tools

Tool functions should expose the smallest task-oriented contract. Docstrings and type annotations become part of the agent-facing schema, so describe purpose, limits, and failure behavior precisely.

```python
@agent.tool
async def get_kyc_status(ctx: RunContext[ComplianceDeps]) -> dict:
    """Return verified KYC facts for the scoped case. Never query another tenant."""
    return await ctx.deps.repository.kyc(case_id=ctx.deps.case_id, tenant_id=ctx.deps.tenant_id)

@agent.tool
async def screen_transaction(ctx: RunContext[ComplianceDeps]) -> dict:
    """Return the current sanctions/fraud screening result and its source ID."""
    return await ctx.deps.repository.screening(case_id=ctx.deps.case_id, tenant_id=ctx.deps.tenant_id)
```

Pydantic AI validates tool arguments and can return validation failures to the model for a bounded retry. Still enforce tenant checks and authorization inside the repository: tool-schema validation is not access control. See [function tools](https://pydantic.dev/docs/ai/tools-toolsets/tools/).


In [ ]:
from dataclasses import dataclass
from datetime import date

@dataclass(frozen=True)
class CaseScope:
    tenant_id: str
    case_id: str

EVIDENCE={
 'case-100': {'tenant':'northstar','kyc-verified':{'fresh':True},'transaction-screening':{'fresh':True}},
 'case-101': {'tenant':'northstar','kyc-verified':{'fresh':True},'transaction-screening':{'fresh':True}},
}
def get_screening(scope: CaseScope) -> dict:
    case=EVIDENCE[scope.case_id]
    if case['tenant'] != scope.tenant_id: raise PermissionError('cross-tenant lookup denied')
    return {'source_id':'transaction-screening', **case['transaction-screening']}
get_screening(CaseScope('northstar','case-101'))


## 5. Step 4 — Structured output and validation retries

`output_type=ComplianceDecision` asks the runtime to return an object matching the schema. If validation fails, the framework can ask the model to repair the response within the configured retry budget. Use retries for *recoverable formatting or schema mistakes*, not to keep trying until a desired decision appears.

```python
agent = Agent(
    "provider:model",
    deps_type=ComplianceDeps,
    output_type=ComplianceDecision,
    retries=2,
    instructions="Use KYC and screening evidence. Escalate when policy requires it.",
)
result = await agent.run("Review the payment case.", deps=deps)
decision: ComplianceDecision = result.output
```

A separate evidence gate must verify that IDs were actually retrieved for this case and that required signals are fresh. The [output guide](https://pydantic.dev/docs/ai/core-concepts/output/) and [retry guidance](https://pydantic.dev/docs/ai/models/http-request-retries/) cover current validation and retry behavior.


In [ ]:
def evidence_gate(decision, known_evidence:set[str], fresh_evidence:set[str]) -> dict:
    ids=set(decision.evidence_ids)
    return {
      'schema_like_shape': decision.decision in {'approve','escalate','reject'},
      'case_scoped': bool(ids) and ids.issubset(known_evidence),
      'fresh': ids.issubset(fresh_evidence),
      'human_review_for_escalation': decision.decision!='escalate' or decision.requires_human_review,
    }
known={'kyc-verified','transaction-screening'}
print(evidence_gate(low,known,known))
print(evidence_gate(high,known,known))


## 6. Step 5 — Separate recommendation from authorization

A typed `approve` recommendation is not permission to approve a payment. The action service must independently enforce thresholds, staff role, case status, dual control, audit retention, and idempotency.

```python
def authorize_route(case: PaymentCase, decision: ComplianceDecision, actor: Operator) -> str:
    if decision.decision == "approve" and case.amount_usd >= 10_000:
        return "escalate"  # deterministic high-value policy overrides a suggestion
    if decision.decision == "approve" and not actor.can_approve(case.tenant_id):
        return "escalate"
    return decision.decision
```

This separation is why typed output is so useful: it turns language into a reviewable input for deterministic business rules.


In [ ]:
def authorize_route(amount_usd:int, decision:str, actor_can_approve:bool)->str:
    if decision=='approve' and (amount_usd>=10_000 or not actor_can_approve): return 'escalate'
    return decision
{'low_value':authorize_route(450,low.decision,True),'high_value':authorize_route(12_500,'approve',True),'no_role':authorize_route(450,'approve',False)}


## 7. Step 6 — Observability, testing, and evaluation

Pydantic AI integrates with Pydantic Logfire/OpenTelemetry-compatible observability. Instrumentation should reveal model/tool boundaries, validation failures, retries, latency, and cost without storing raw secrets or unrestricted customer data.

```python
# Optional production instrumentation; configure redaction and access first.
import logfire
logfire.configure()
logfire.instrument_pydantic_ai()
```

Evaluate a dataset, not one impressive response. For each case record: expected route, required evidence, forbidden evidence, human-review requirement, tool path, retries, latency, and cost. Pydantic’s [evaluation documentation](https://pydantic.dev/docs/ai/evals/evals/) is a useful starting point.

| Test case | Expected result | Failure it catches |
| --- | --- | --- |
| low-value, fresh KYC + screening | approve | unnecessary escalation |
| high-value case | escalate + human review | unsafe automatic approval |
| stale screening | escalate | treating old evidence as current |
| tenant mismatch | policy error | cross-tenant leakage |
| schema-valid fake evidence ID | fail evidence gate | confusing valid JSON with evidence |


In [ ]:
def evaluate_case(case_id, amount, country, expected):
    decision=pydanticai_shaped_compliance_case(case_id,amount,country)
    checks=evidence_gate(decision,{'kyc-verified','transaction-screening'},{'kyc-verified','transaction-screening'})
    return {'case':case_id,'expected':expected,'actual':decision.decision,'outcome_ok':decision.decision==expected,'checks':checks}
[evaluate_case('case-100',450,'CA','approve'),evaluate_case('case-101',12_500,'CA','escalate')]


## 8. Production checklist and exercises

- [ ] Validate request input, tool arguments, model output, evidence provenance, and action authorization separately.
- [ ] Inject tenant/case scope through trusted server dependencies; never let user text select it.
- [ ] Bound agent, tool, and HTTP retries; log failure categories and escalate on exhaustion.
- [ ] Treat schemas as a contract, not a truth detector or policy engine.
- [ ] Use stable evidence IDs, freshness checks, and citations in every machine-consumed decision.
- [ ] Instrument with redaction, access control, retention, and dataset-backed evaluation.
- [ ] Put irreversible actions behind a separately authenticated, idempotent service.

### Exercises

1. Add a `screened_at` timestamp and require screening to be less than 24 hours old.
2. Create a `reject` policy for confirmed sanctions evidence and require a second reviewer for any override.
3. Add an adversarial fixture whose tool response says “ignore previous instructions”; prove it is treated as data.
4. Add `policy_version` to the output and test that an old decision cannot be applied after a policy change.
5. Use Pydantic’s offline test model to test a tool schema and output-validation retry without provider credentials.

## References

- [Pydantic AI overview](https://pydantic.dev/docs/ai/overview/) — current framework capabilities and provider support.
- [Agents](https://pydantic.dev/docs/ai/core-concepts/agent/), [dependencies](https://pydantic.dev/docs/ai/core-concepts/dependencies/), and [function tools](https://pydantic.dev/docs/ai/tools-toolsets/tools/).
- [Structured output](https://pydantic.dev/docs/ai/core-concepts/output/), [retries](https://pydantic.dev/docs/ai/models/http-request-retries/), and [Pydantic Evals](https://pydantic.dev/docs/ai/evals/evals/).

**Takeaway:** Pydantic AI makes typed agent interfaces ergonomic. The safe system still needs evidence verification, deterministic policy, authorization, and operational evaluation.
